# Agent SQL v2 : conçu pour un catalogue sale (naming, tables périmées, peu de métadonnées)

Suite de `sql_agent.ipynb`, avec un objectif différent : pas explorer plusieurs
architectures, mais **en construire une qui tienne sur un vrai catalogue** — celui
qu'on a, pas celui qu'on voudrait. Trois contraintes structurent tout ce notebook :

1. **Beaucoup d'assets, dont des périmés.** Un catalogue réel a des tables `_BCK`,
   `_OLD`, `_TMP`, `_TEST`, des copies de sauvegarde au nom presque identique à la
   table légitime. Un agent qui se contente de la recherche sémantique s'y fait
   piéger — le nom d'une sauvegarde ressemble autant à la question que l'original.
2. **Naming incohérent, peu de conventions.** Certaines tables sont bien nommées et
   documentées, d'autres ont des colonnes cryptiques (`c1`, `c2`...) sans commentaire.
3. **Très peu de métadonnées.** La plupart des tables/colonnes n'ont pas de
   description — il faut une autre source de signal.

**Le choix d'architecture** : un pipeline **guidé mais étroit** (le code décide de
l'ordre des étapes et des garde-fous, le modèle ne remplit que des cases précises),
plutôt qu'un agent en tool-calling libre. Sur un catalogue propre et petit (v1), la
différence est marginale. Sur un catalogue sale, elle ne l'est pas : le code peut
appliquer des règles que la sémantique seule ne voit pas ("ce nom ressemble à une
sauvegarde, l'exclure avant même que le modèle le voie"), là où un agent libre doit
deviner à chaque tour.

**Le graphe** (section 5) : `retrieve_candidates` (recherche + filtre les tables
périmées + trie par un score de confiance) → `gather_context` (métadonnées +
échantillonnage des colonnes non documentées, pour compenser l'absence de
description) → `generate_sql` (1 appel LLM, JSON contraint, liste explicite des
tables autorisées) → `validate_and_execute` (SELECT seul, tables/colonnes connues,
rejette les tables périmées même si le modèle insiste, relance une fois sur un
résultat vide) → `finalize` (réponse + table(s) citée(s)) ou `abstain`.

**Un seul graphe, paramétré par un flag `hardening`** — pas trois pipelines
recodés : `hardening=False` reproduit le pipeline naïf de v1 (aucune conscience de
la dette du catalogue), `hardening=True` active les trois défenses ci-dessus. La
section 7 compare ce pipeline (dans les deux configurations) à un agent libre
(section 6, repris de v1) sur 12 questions, jugées par un LLM contre une grille
d'éléments attendus — pas une réponse exacte.

In [ ]:
from __future__ import annotations

import json
import logging
import os
import re
import sqlite3
import threading
import time
from pathlib import Path

import numpy as np
import sqlglot
from dotenv import load_dotenv
from openai import OpenAI
from sqlglot import exp

load_dotenv()
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
log = logging.getLogger("sql_agent_v2")

# ============================================================================
# CONFIGURATION -- tout ce qui est tweakable est ici.
# ============================================================================

PROVIDER = "ollama"  # "openrouter" (cloud) | "ollama" (local) -- meme code, meme API
OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "anthropic/claude-haiku-4.5")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")  # doit supporter le tool calling (approche agent libre)
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")

DB_BACKEND = "demo_sqlite"  # "demo_sqlite" (aucune infra) | "oracle" (datamart reel)
ORACLE_DSN = os.environ.get("ORACLE_DSN", "hote:1521/service")
ORACLE_USER = os.environ.get("ORACLE_USER", "compte_lecture")

MAX_RESULT_ROWS = 50
QUERY_TIMEOUT_SECONDS = 10
MAX_TOOL_ROUNDS = 8           # borne de la boucle agent libre (section 6)
MAX_SQL_ATTEMPTS = 3          # borne des tentatives de generation SQL (pipeline guide, section 5)
SEMANTIC_TOP_K = 8            # candidats bruts avant filtrage/tri (hardening) ou troncature (naif)
CANDIDATE_SHORTLIST_SIZE = 4  # tables effectivement montrees au modele pour generer le SQL

# Heuristique de peremption par nom -- sans metadonnee fiable de fraicheur (pas de
# LAST_ANALYZED synthetique ici), c'est le signal le moins cher et le plus robuste :
# les tables de sauvegarde/temp/test suivent presque toujours une convention de suffixe.
DEPRECATED_NAME_PATTERN = re.compile(r"(_BCK|_OLD|_TMP|_COPY|_TEST|_V\d+|_\d{4})$", re.IGNORECASE)

# Le juge du benchmark (section 7) peut etre un modele different de l'agent, pour
# limiter le biais d'auto-evaluation -- par defaut le meme, par simplicite.
JUDGE_PROVIDER = PROVIDER
JUDGE_MODEL_OVERRIDE: str | None = None  # ex. "anthropic/claude-sonnet-4.5" si PROVIDER == "openrouter"


def get_client_and_model(provider: str) -> tuple[OpenAI, str]:
    if provider == "openrouter":
        return (
            OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["LLM_API_KEY"]),
            OPENROUTER_MODEL,
        )
    if provider == "ollama":
        return OpenAI(base_url=f"{OLLAMA_HOST}/v1", api_key="ollama"), OLLAMA_MODEL
    raise ValueError(f"provider inconnu : {provider!r}")


client, MODEL = get_client_and_model(PROVIDER)
judge_client, JUDGE_MODEL = get_client_and_model(JUDGE_PROVIDER)
JUDGE_MODEL = JUDGE_MODEL_OVERRIDE or JUDGE_MODEL

## 1. Un catalogue volontairement sale

12 tables : 8 « légitimes » (5 documentées comme en v1, 3 pas du tout — dont une aux
colonnes cryptiques `c1`..`c5`, le genre de table de staging qu'on trouve toujours en
vrai), et **4 pièges** — des noms qui ressemblent fortement à une table légitime mais
qui sont des sauvegardes/copies/tests, avec des données volontairement périmées ou
incohérentes (`dmt_cpt_sld_j_old` contient un solde différent de la vraie table). Le
but : que les questions de la section 7 puissent vérifier si une approche se fait
piéger, pas seulement si elle "trouve une table qui matche".

In [ ]:
CATALOG: dict[str, dict] = {
    # -- Legitimes, documentees --------------------------------------------------
    "DMT.DMT_CPT_MVT_J": {
        "sqlite_table": "dmt_cpt_mvt_j",
        "comment": "Mouvements comptables journaliers par compte.",
        "columns": [
            {"name": "id_mvt", "type": "NUMBER", "key": "PK", "comment": ""},
            {"name": "id_compte", "type": "NUMBER", "key": "FK", "comment": "-> comptes (implicite)"},
            {"name": "dt_mvt", "type": "DATE", "key": "", "comment": ""},
            {"name": "cd_typ_ope", "type": "VARCHAR2", "key": "", "comment": ""},
            {"name": "mt_mvt", "type": "NUMBER", "key": "", "comment": "Montant signe, devise d'origine."},
        ],
    },
    "DMT.DMT_CPT_SLD_J": {
        "sqlite_table": "dmt_cpt_sld_j",
        "comment": "Solde de fin de journee par compte, en CHF (alimente par les mouvements, voir lineage).",
        "columns": [
            {"name": "id_compte", "type": "NUMBER", "key": "FK", "comment": ""},
            {"name": "dt_jour", "type": "DATE", "key": "", "comment": ""},
            {"name": "mt_sld_chf", "type": "NUMBER", "key": "", "comment": "Cumul des mouvements, voir lineage."},
        ],
    },
    "ODS.ODS_D_CLI_ADR": {
        "sqlite_table": "ods_d_cli_adr",
        "comment": "Adresses postales des clients.",
        "columns": [
            {"name": "id_client", "type": "NUMBER", "key": "PK", "comment": ""},
            {"name": "rue", "type": "VARCHAR2", "key": "", "comment": ""},
            {"name": "ville", "type": "VARCHAR2", "key": "", "comment": ""},
            {"name": "dt_deb_val", "type": "DATE", "key": "", "comment": ""},
        ],
    },
    "DMT.DMT_F_CRD_ENC_M": {
        "sqlite_table": "dmt_f_crd_enc_m",
        "comment": "Encours de credit mensuels par dossier.",
        "columns": [
            {"name": "id_dossier", "type": "NUMBER", "key": "PK", "comment": ""},
            {"name": "id_client", "type": "NUMBER", "key": "FK", "comment": ""},
            {"name": "dt_fin_mois", "type": "DATE", "key": "", "comment": ""},
            {"name": "mt_crd_restant", "type": "NUMBER", "key": "", "comment": "Capital restant du en fin de mois."},
        ],
    },
    "REF.V_REF_FIN_TXC_CHF": {
        "sqlite_table": "v_ref_fin_txc_chf",
        "comment": "Cours de conversion quotidiens vers le franc suisse.",
        "columns": [
            {"name": "devise", "type": "VARCHAR2", "key": "", "comment": ""},
            {"name": "dt_jour", "type": "DATE", "key": "", "comment": ""},
            {"name": "tx_chf", "type": "NUMBER", "key": "", "comment": ""},
        ],
    },
    # -- Legitimes, NON documentees (le regime realiste) -------------------------
    "ODS.ODS_D_CLI": {
        "sqlite_table": "ods_d_cli",
        "comment": "",
        "columns": [
            {"name": "id_client", "type": "NUMBER", "key": "PK", "comment": ""},
            {"name": "nom", "type": "VARCHAR2", "key": "", "comment": ""},
            {"name": "prenom", "type": "VARCHAR2", "key": "", "comment": ""},
            {"name": "segment", "type": "VARCHAR2", "key": "", "comment": ""},
        ],
    },
    "REF.R_TYP_OPE": {
        "sqlite_table": "r_typ_ope",
        "comment": "",
        "columns": [
            {"name": "cd_typ_ope", "type": "VARCHAR2", "key": "PK", "comment": ""},
            {"name": "lib_typ_ope", "type": "VARCHAR2", "key": "", "comment": ""},
        ],
    },
    "STG.STG_CPT_RAW": {
        "sqlite_table": "stg_cpt_raw",
        "comment": "",  # staging brut, jamais documente -- bruit realiste pour la recherche
        "columns": [
            {"name": "c1", "type": "NUMBER", "key": "", "comment": ""},
            {"name": "c2", "type": "VARCHAR2", "key": "", "comment": ""},
            {"name": "c3", "type": "NUMBER", "key": "", "comment": ""},
        ],
    },
    # -- Pieges : noms proches d'une table legitime, perimes ---------------------
    "ODS.ODS_F_CPT_MVT_BCK_2019": {
        "sqlite_table": "ods_f_cpt_mvt_bck_2019",
        "comment": "",
        "columns": [
            {"name": "id_mvt", "type": "NUMBER", "key": "", "comment": ""},
            {"name": "id_compte", "type": "NUMBER", "key": "", "comment": ""},
            {"name": "mt_mvt", "type": "NUMBER", "key": "", "comment": ""},
        ],
    },
    "DMT.DMT_CPT_SLD_J_OLD": {
        "sqlite_table": "dmt_cpt_sld_j_old",
        "comment": "",
        "columns": [
            {"name": "id_compte", "type": "NUMBER", "key": "", "comment": ""},
            {"name": "mt_sld_chf", "type": "NUMBER", "key": "", "comment": ""},
        ],
    },
    "STG.STG_CPT_SLD_TMP_COPY": {
        "sqlite_table": "stg_cpt_sld_tmp_copy",
        "comment": "",
        "columns": [
            {"name": "id_compte", "type": "NUMBER", "key": "", "comment": ""},
            {"name": "mt_sld_chf", "type": "NUMBER", "key": "", "comment": ""},
        ],
    },
    "TEC.TEC_CPT_MVT_TEST": {
        "sqlite_table": "tec_cpt_mvt_test",
        "comment": "",
        "columns": [
            {"name": "id_compte", "type": "NUMBER", "key": "", "comment": ""},
            {"name": "mt_mvt", "type": "NUMBER", "key": "", "comment": ""},
        ],
    },
}

_SQLITE_TO_FQN = {meta["sqlite_table"]: fqn for fqn, meta in CATALOG.items()}


def is_deprecated_name(sqlite_table_name: str) -> bool:
    return bool(DEPRECATED_NAME_PATTERN.search(sqlite_table_name.upper()))

In [ ]:
def build_demo_database() -> sqlite3.Connection:
    conn = sqlite3.connect(":memory:", check_same_thread=False)
    conn.executescript("""
        CREATE TABLE dmt_cpt_mvt_j (id_mvt INTEGER PRIMARY KEY, id_compte INTEGER, dt_mvt TEXT, cd_typ_ope TEXT, mt_mvt REAL);
        CREATE TABLE dmt_cpt_sld_j (id_compte INTEGER, dt_jour TEXT, mt_sld_chf REAL);
        CREATE TABLE ods_d_cli_adr (id_client INTEGER, rue TEXT, ville TEXT, dt_deb_val TEXT);
        CREATE TABLE dmt_f_crd_enc_m (id_dossier INTEGER, id_client INTEGER, dt_fin_mois TEXT, mt_crd_restant REAL);
        CREATE TABLE v_ref_fin_txc_chf (devise TEXT, dt_jour TEXT, tx_chf REAL);
        CREATE TABLE ods_d_cli (id_client INTEGER PRIMARY KEY, nom TEXT, prenom TEXT, segment TEXT);
        CREATE TABLE r_typ_ope (cd_typ_ope TEXT PRIMARY KEY, lib_typ_ope TEXT);
        CREATE TABLE stg_cpt_raw (c1 INTEGER, c2 TEXT, c3 REAL);
        CREATE TABLE ods_f_cpt_mvt_bck_2019 (id_mvt INTEGER, id_compte INTEGER, mt_mvt REAL);
        CREATE TABLE dmt_cpt_sld_j_old (id_compte INTEGER, mt_sld_chf REAL);
        CREATE TABLE stg_cpt_sld_tmp_copy (id_compte INTEGER, mt_sld_chf REAL);
        CREATE TABLE tec_cpt_mvt_test (id_compte INTEGER, mt_mvt REAL);
    """)
    rows = {
        "dmt_cpt_mvt_j": [
            (1, 101, "2026-09-01", "debit", -120.50), (2, 101, "2026-09-02", "credit", 500.00),
            (3, 102, "2026-09-01", "debit", -40.00), (4, 102, "2026-09-03", "debit", -15.90),
            (5, 103, "2026-09-02", "credit", 1200.00), (6, 101, "2026-09-05", "debit", -60.00),
            (7, 102, "2026-09-05", "virement", -200.00), (8, 103, "2026-09-06", "debit", -75.30),
        ],
        "dmt_cpt_sld_j": [
            (101, "2026-09-05", 2340.75), (102, "2026-09-05", 890.10), (103, "2026-09-05", 15420.00),
        ],
        "ods_d_cli_adr": [
            (201, "Rue du Lac 4", "Geneve", "2020-01-01"),
            (202, "Bahnhofstrasse 12", "Zurich", "2019-06-15"),
            (203, "Via Nassa 9", "Lugano", "2021-03-10"),
        ],
        "dmt_f_crd_enc_m": [
            (301, 201, "2026-08-31", 45000.00), (302, 202, "2026-08-31", 128900.00),
            (303, 203, "2026-08-31", 0.0),
        ],
        "v_ref_fin_txc_chf": [
            ("EUR", "2026-09-05", 0.96), ("USD", "2026-09-05", 0.88), ("CHF", "2026-09-05", 1.0),
        ],
        "ods_d_cli": [
            (201, "Dupont", "Marie", "particulier"), (202, "Muller", "Hans", "particulier"),
            (203, "Rossi", "Elena", "professionnel"),
        ],
        "r_typ_ope": [("debit", "Debit"), ("credit", "Credit"), ("virement", "Virement")],
        "stg_cpt_raw": [(1, "x", 0.1), (2, "y", 0.2)],
        # Pieges : donnees perimees/incoherentes par rapport aux tables legitimes ci-dessus.
        "ods_f_cpt_mvt_bck_2019": [(1, 101, -30.0)],
        "dmt_cpt_sld_j_old": [(101, 999.99), (102, 100.00)],
        "stg_cpt_sld_tmp_copy": [],
        "tec_cpt_mvt_test": [(999, 1.0)],
    }
    for table, values in rows.items():
        if not values:
            continue
        placeholders = ", ".join("?" * len(values[0]))
        conn.executemany(f"INSERT INTO {table} VALUES ({placeholders})", values)
    conn.commit()
    return conn


DEMO_DB = build_demo_database() if DB_BACKEND == "demo_sqlite" else None

LINEAGE_COLUMNS = [
    {
        "target": "DMT.DMT_CPT_SLD_J.MT_SLD_CHF",
        "expression": "SUM(m.mt_mvt) OVER (PARTITION BY m.id_compte ORDER BY m.dt_mvt)",
        "sources": ["DMT.DMT_CPT_MVT_J.MT_MVT", "DMT.DMT_CPT_MVT_J.ID_COMPTE"],
        "object_fqn": "DMT.DMT_CPT_SLD_J",
    },
]
JOIN_EDGES = [
    {"left": "DMT.DMT_CPT_MVT_J.ID_COMPTE", "right": "DMT.DMT_CPT_SLD_J.ID_COMPTE", "frequency": 1},
    {"left": "DMT.DMT_F_CRD_ENC_M.ID_CLIENT", "right": "ODS.ODS_D_CLI_ADR.ID_CLIENT", "frequency": 1},
    {"left": "DMT.DMT_F_CRD_ENC_M.ID_CLIENT", "right": "ODS.ODS_D_CLI.ID_CLIENT", "frequency": 1},
]

## 2. Outils

Mêmes quatre outils que v1 (`search_catalog`, `get_table_metadata`, `get_lineage`,
`run_sql`), plus un nouveau : **`profile_column`**, qui échantillonne les valeurs
d'une colonne. Sur un catalogue peu documenté, c'est souvent la seule façon de
savoir ce que contient une colonne (`cd_typ_ope` ne dit rien tant qu'on n'a pas vu
qu'il contient `"debit"`/`"credit"`/`"virement"`) — c'est le même principe que
`docmaker/pipeline/profile.py` (`ValueProfile`/`TopValue`) du dépôt, réduit à
l'essentiel pour ce notebook.

In [ ]:
from fastembed import TextEmbedding

_EMBEDDER = TextEmbedding("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


def _catalog_documents() -> list[dict]:
    docs = []
    for fqn, meta in CATALOG.items():
        cols = ", ".join(c["name"] for c in meta["columns"])
        docs.append({"fqn": fqn, "entity_type": "table",
                      "text": f"table {fqn} ({meta['comment']}) colonnes: {cols}"})
        for c in meta["columns"]:
            docs.append({"fqn": f"{fqn}.{c['name']}", "entity_type": "column",
                          "text": f"colonne {c['name']} de {fqn} : {c['comment'] or c['type']}"})
    return docs


_CATALOG_DOCS = _catalog_documents()
_CATALOG_VECTORS = np.array(list(_EMBEDDER.embed([d["text"] for d in _CATALOG_DOCS])))
_CATALOG_VECTORS = _CATALOG_VECTORS / np.linalg.norm(_CATALOG_VECTORS, axis=1, keepdims=True)


def search_catalog(query: str, top_k: int = SEMANTIC_TOP_K, entity_type: str | None = None) -> list[dict]:
    vector = next(iter(_EMBEDDER.embed([query])))
    vector = vector / np.linalg.norm(vector)
    scores = _CATALOG_VECTORS @ vector
    order = np.argsort(-scores)
    hits = []
    for i in order:
        doc = _CATALOG_DOCS[i]
        if entity_type and doc["entity_type"] != entity_type:
            continue
        hits.append({"fqn": doc["fqn"], "entity_type": doc["entity_type"], "score": round(float(scores[i]), 3)})
        if len(hits) >= top_k:
            break
    return hits


def get_table_metadata(fqn: str) -> dict:
    meta = CATALOG.get(fqn.upper())
    if meta is None:
        return {"error": f"table inconnue : {fqn!r}. Tables disponibles : {list(CATALOG)}"}
    row_count = None
    if DB_BACKEND == "demo_sqlite":
        row_count = DEMO_DB.execute(f"SELECT COUNT(*) FROM {meta['sqlite_table']}").fetchone()[0]
    return {"fqn": fqn.upper(), "comment": meta["comment"], "row_count": row_count, "columns": meta["columns"]}


def get_lineage(table_or_column: str) -> dict:
    needle = table_or_column.upper()
    columns = [c for c in LINEAGE_COLUMNS if needle in c["target"] or any(needle in s for s in c["sources"])]
    joins = [j for j in JOIN_EDGES if needle in j["left"] or needle in j["right"]]
    if not columns and not joins:
        return {"message": f"aucun lineage trouve pour {table_or_column!r}"}
    return {"column_lineage": columns, "joins": joins}


def profile_column(fqn: str, column: str, top_n: int = 5) -> dict:
    """Distribution des valeurs les plus frequentes d'une colonne -- compense
    l'absence de description sur un catalogue peu documente. `column` est valide
    contre le schema connu avant interpolation SQL (argument fourni par le modele).
    """
    meta = CATALOG.get(fqn.upper())
    if meta is None:
        return {"error": f"table inconnue : {fqn!r}"}
    valid_columns = {c["name"] for c in meta["columns"]}
    if column not in valid_columns:
        return {"error": f"colonne inconnue sur {fqn} : {column!r}. Colonnes : {sorted(valid_columns)}"}
    table = meta["sqlite_table"]
    rows = DEMO_DB.execute(
        f"SELECT {column} AS v, COUNT(*) AS n FROM {table} GROUP BY {column} ORDER BY n DESC LIMIT ?",
        (top_n,),
    ).fetchall()
    total = DEMO_DB.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    return {
        "column": column,
        "top_values": [{"value": str(v), "frequency": round(n / total, 3) if total else 0.0} for v, n in rows],
    }

In [ ]:
class ReadOnlyViolation(Exception):
    pass


def _ensure_select_only(sql: str, dialect: str) -> None:
    try:
        statements = sqlglot.parse(sql, dialect=dialect)
    except Exception as e:
        raise ReadOnlyViolation(f"SQL non parsable : {e}") from e
    if len(statements) != 1 or statements[0] is None:
        raise ReadOnlyViolation("une seule instruction a la fois")
    if not isinstance(statements[0], exp.Select):
        raise ReadOnlyViolation("seules les requetes SELECT sont autorisees")


def _cap_rows_sqlite(sql: str, limit: int) -> str:
    return f"SELECT * FROM ({sql}) AS capped LIMIT {limit}"


def _run_sqlite(sql: str, limit: int, timeout_s: float) -> dict:
    result: dict = {}

    def worker():
        try:
            cur = DEMO_DB.execute(_cap_rows_sqlite(sql, limit))
            cols = [d[0] for d in cur.description]
            result["columns"] = cols
            result["rows"] = [dict(zip(cols, r, strict=True)) for r in cur.fetchall()]
        except Exception as e:  # noqa: BLE001
            result["error"] = str(e)

    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    thread.join(timeout_s)
    if thread.is_alive():
        return {"error": f"depassement du budget de {timeout_s}s (backend demo, non annulable)"}
    return result


def _run_oracle(sql: str, limit: int, timeout_s: float) -> dict:
    import oracledb

    capped = f"SELECT * FROM ({sql}) FETCH FIRST {limit} ROWS ONLY"
    password = os.environ.get("ORACLE_PASSWORD")
    if not password:
        return {"error": "ORACLE_PASSWORD absent de l'environnement (.env)"}
    try:
        with oracledb.connect(user=ORACLE_USER, password=password, dsn=ORACLE_DSN) as conn:
            conn.call_timeout = int(timeout_s * 1000)
            with conn.cursor() as cur:
                cur.execute(capped)
                cols = [d[0].lower() for d in cur.description]
                rows = [dict(zip(cols, r, strict=True)) for r in cur.fetchall()]
                return {"columns": cols, "rows": rows}
    except Exception as e:  # noqa: BLE001
        return {"error": str(e)}


def run_sql(sql: str) -> dict:
    dialect = "oracle" if DB_BACKEND == "oracle" else "sqlite"
    try:
        _ensure_select_only(sql, dialect)
    except ReadOnlyViolation as e:
        return {"error": str(e)}

    if DB_BACKEND == "demo_sqlite":
        outcome = _run_sqlite(sql, MAX_RESULT_ROWS, QUERY_TIMEOUT_SECONDS)
    elif DB_BACKEND == "oracle":
        outcome = _run_oracle(sql, MAX_RESULT_ROWS, QUERY_TIMEOUT_SECONDS)
    else:
        raise ValueError(f"DB_BACKEND inconnu : {DB_BACKEND!r}")

    if "error" in outcome:
        return outcome
    truncated = len(outcome["rows"]) >= MAX_RESULT_ROWS
    return {"columns": outcome["columns"], "rows": outcome["rows"], "truncated": truncated}

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "search_catalog",
        "description": "Recherche semantique de tables/colonnes pertinentes pour une question en langage naturel.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}, "top_k": {"type": "integer", "default": SEMANTIC_TOP_K},
            "entity_type": {"type": "string", "enum": ["table", "column"]},
        }, "required": ["query"]},
    }},
    {"type": "function", "function": {
        "name": "get_table_metadata",
        "description": "Colonnes, types, cles et commentaire d'une table (FQN 'SCHEMA.TABLE').",
        "parameters": {"type": "object", "properties": {"fqn": {"type": "string"}}, "required": ["fqn"]},
    }},
    {"type": "function", "function": {
        "name": "get_lineage",
        "description": "Expression de calcul d'une colonne derivee et jointures connues pour une table ou colonne.",
        "parameters": {"type": "object", "properties": {"table_or_column": {"type": "string"}}, "required": ["table_or_column"]},
    }},
    {"type": "function", "function": {
        "name": "profile_column",
        "description": "Valeurs les plus frequentes d'une colonne -- utile quand son nom ou son type ne suffit pas a comprendre son contenu.",
        "parameters": {"type": "object", "properties": {
            "fqn": {"type": "string"}, "column": {"type": "string"}, "top_n": {"type": "integer", "default": 5},
        }, "required": ["fqn", "column"]},
    }},
    {"type": "function", "function": {
        "name": "run_sql",
        "description": (
            f"Execute une requete SQL en LECTURE SEULE (SELECT uniquement, plafonnee a {MAX_RESULT_ROWS} lignes). "
            "A utiliser aussi bien pour explorer les donnees que pour calculer la reponse finale a la question."
        ),
        "parameters": {"type": "object", "properties": {"sql": {"type": "string"}}, "required": ["sql"]},
    }},
]

TOOL_FUNCS = {
    "search_catalog": search_catalog, "get_table_metadata": get_table_metadata,
    "get_lineage": get_lineage, "profile_column": profile_column, "run_sql": run_sql,
}


def dispatch_tool_call(name: str, arguments: dict) -> str:
    func = TOOL_FUNCS.get(name)
    if func is None:
        return json.dumps({"error": f"outil inconnu : {name!r}"})
    try:
        return json.dumps(func(**arguments), default=str, ensure_ascii=False)
    except Exception as e:  # noqa: BLE001
        return json.dumps({"error": str(e)}, ensure_ascii=False)

## 3. Signal de confiance : repérer ce qui est périmé sans métadonnée fiable

Sans `LAST_ANALYZED` ni tag de classification fiable (le régime réaliste), le
signal le moins cher est le **nom** : les tables de sauvegarde/temp/test suivent
presque toujours une convention de suffixe (`_BCK`, `_OLD`, `_TMP`, `_TEST`, un
numéro de version, une année). `is_deprecated_name` (section 1) l'exploite.

`trust_score` combine ce signal avec ce que le catalogue sait déjà faire sans LLM :
présence d'un commentaire, d'une clé primaire, de lignes non vides. Même esprit que
`docmaker/pipeline/priority.py` du dépôt (classement par signaux disponibles,
jamais un signal manquant mis à zéro), réduit à quelques règles pour ce notebook.

In [ ]:
def trust_score(fqn: str) -> float:
    meta = CATALOG[fqn]
    score = 0.0
    if meta["comment"]:
        score += 2.0
    documented_ratio = sum(1 for c in meta["columns"] if c["comment"]) / max(len(meta["columns"]), 1)
    score += documented_ratio
    if any(c["key"] == "PK" for c in meta["columns"]):
        score += 1.0
    if is_deprecated_name(meta["sqlite_table"]):
        score -= 5.0
    if DB_BACKEND == "demo_sqlite":
        row_count = DEMO_DB.execute(f"SELECT COUNT(*) FROM {meta['sqlite_table']}").fetchone()[0]
        if row_count == 0:
            score -= 2.0
    return score

## 4. `_extract_json_object` — sortie JSON tolérante

Même approche que `docmaker/llm.py::_strip` du dépôt : on ne suppose pas que le
modèle respecte un mode JSON strict (variable selon le backend OpenAI-compatible),
on nettoie un éventuel bloc de code et on isole le premier objet. Réutilisé par
`generate_sql` (section 5) et par le juge du benchmark (section 7).

In [ ]:
_JSON_FENCE = "```"


def _extract_json_object(raw: str) -> dict:
    s = raw.strip()
    if s.startswith(_JSON_FENCE):
        s = s.strip("`").removeprefix("json").strip()
    i, j = s.find("{"), s.rfind("}")
    return json.loads(s[i : j + 1] if 0 <= i < j else s)

## 5. Le pipeline guidé, paramétré par un flag `hardening`

Un seul graphe LangGraph. Chaque nœud regarde `state["hardening"]` pour décider
d'appliquer ou non les trois défenses ; `hardening=False` reproduit le pipeline
naïf de v1 (aucune conscience de la dette du catalogue), `hardening=True` les
active toutes. Ça évite de recoder trois pipelines pour le benchmark (section 7).

In [ ]:
try:
    import operator
    from typing import Annotated, TypedDict

    from langgraph.graph import END, START, StateGraph
except ImportError as e:
    raise ImportError(
        "LangGraph n'est pas installe dans cet environnement. "
        "Poetry : `poetry add langgraph`. Sinon : `pip install langgraph`."
    ) from e


class PipelineState(TypedDict):
    question: str
    hardening: bool
    attempts: int
    shortlist: list[str]
    context: str
    sql: str
    error: str
    zero_row_retried: bool
    rows: list[dict]
    answer: str

### `retrieve_candidates` — filtrer les pièges avant que le modèle les voie

Défense n°1 : si `hardening`, les tables au nom périmé sont retirées de la liste
*avant* que le LLM ne la voie (il ne peut pas être piégé par ce qu'on ne lui montre
jamais), et le reste est trié par `trust_score`. Sans `hardening`, on garde le
classement brut de la recherche sémantique — exactement ce qui expose au piège.

In [ ]:
def retrieve_candidates(state: PipelineState) -> dict:
    hits = search_catalog(state["question"], top_k=SEMANTIC_TOP_K, entity_type="table")
    fqns = [h["fqn"] for h in hits]
    if state["hardening"]:
        fqns = [f for f in fqns if not is_deprecated_name(CATALOG[f]["sqlite_table"])]
        fqns.sort(key=trust_score, reverse=True)
    shortlist = fqns[:CANDIDATE_SHORTLIST_SIZE]
    return {"shortlist": shortlist, "attempts": 0, "error": "", "zero_row_retried": False}

### `gather_context` — compenser l'absence de description par l'échantillonnage

Défense n°2 : pour chaque colonne texte sans commentaire d'une table retenue (si
`hardening`), on appelle `profile_column` et on injecte les valeurs observées dans
le contexte — c'est souvent la seule façon de savoir que `cd_typ_ope` contient
`"debit"/"credit"/"virement"`. Sans `hardening`, le modèle ne voit que les noms de
colonnes bruts, comme en v1.

In [ ]:
def gather_context(state: PipelineState) -> dict:
    lines = []
    for fqn in state["shortlist"]:
        meta = CATALOG[fqn]
        col_lines = []
        for c in meta["columns"]:
            desc = c["comment"]
            if not desc and state["hardening"] and c["type"] == "VARCHAR2":
                sample = profile_column(fqn, c["name"], top_n=3)
                values = sample.get("top_values") or []
                if values:
                    desc = "valeurs observees: " + ", ".join(v["value"] for v in values)
            col_lines.append(f"{c['name']} ({c['type']}{', ' + c['key'] if c['key'] else ''}): {desc or '?'}")
        lines.append(
            f"- table sqlite `{meta['sqlite_table']}` [{fqn}] : {meta['comment'] or '(non documentee)'}\n"
            f"  colonnes: " + "; ".join(col_lines)
        )
    return {"context": "\n".join(lines)}

### `generate_sql` — une seule case à remplir, un périmètre explicite

Le prompt énumère les tables autorisées et prévient explicitement le modèle de ne
pas céder à un nom "plus évident" hors de cette liste — la contrainte est répétée
en langage naturel *et* vérifiée par le code (`validate_and_execute`, juste après) :
dire au modèle ne suffit jamais seul sur un catalogue piégeur.

In [ ]:
def generate_sql(state: PipelineState) -> dict:
    prompt = f"Tables disponibles (utilise UNIQUEMENT celles-ci) :\n{state['context']}\n\nQuestion : {state['question']}\n"
    if state.get("error"):
        prompt += (
            f"\nLa proposition precedente a echoue :\n{state.get('sql', '')}\n"
            f"Erreur : {state['error']}\nCorrige le SQL en consequence."
        )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "Tu ecris une seule requete SQL SQLite, en SELECT uniquement, sur les tables "
                "sqlite listees (utilise leur nom sqlite, pas le FQN entre crochets). N'utilise "
                "jamais une table hors de cette liste, meme si son nom te semble plus evident "
                "ou plus direct. "
                'Reponds avec UN SEUL objet JSON, sans texte autour : {"sql": "..."}'
            )},
            {"role": "user", "content": prompt},
        ],
    )
    raw = response.choices[0].message.content or ""
    try:
        sql = _extract_json_object(raw)["sql"]
    except Exception as e:
        return {"sql": "", "error": f"reponse non JSON valide : {e}", "attempts": state["attempts"] + 1}
    return {"sql": sql, "attempts": state["attempts"] + 1}

### `validate_and_execute` — trois garde-fous, dont deux nouveaux par rapport à v1

1. SELECT seul, tables connues du catalogue (comme v1).
2. **Nouveau, colonnes connues** : chaque colonne référencée doit exister sur au
   moins une des tables de la requête (approximatif — pas de résolution précise par
   alias, mais suffisant pour attraper une colonne inventée sur un nom cryptique).
3. **Nouveau, défense n°3, si `hardening`** : rejette toute table périmée que le
   modèle aurait quand même utilisée (il arrive qu'il l'ignore) — le contrôle final
   ne dépend jamais uniquement du prompt.
4. **Nouveau, si `hardening`** : un résultat vide déclenche **une seule** relance
   avec indice ("vérifie les jointures/filtres") avant d'accepter la réponse vide
   comme réponse finale — un résultat vide est ambigu (mauvais join, ou vraie
   absence de donnée) et mérite une vérification, mais pas une boucle infinie.

In [ ]:
def validate_and_execute(state: PipelineState) -> dict:
    sql = state["sql"]
    if not sql:
        return {"error": state.get("error") or "sql vide"}
    try:
        _ensure_select_only(sql, "sqlite")
    except ReadOnlyViolation as e:
        return {"error": str(e)}

    tree = sqlglot.parse_one(sql, dialect="sqlite")
    referenced_tables = {t.name.lower() for t in tree.find_all(exp.Table)}
    unknown_tables = referenced_tables - set(_SQLITE_TO_FQN)
    if unknown_tables:
        return {"error": f"table(s) inconnue(s) du catalogue : {sorted(unknown_tables)}"}

    if state["hardening"]:
        deprecated_used = {t for t in referenced_tables if is_deprecated_name(t)}
        if deprecated_used:
            return {"error": f"table(s) perimee(s)/de sauvegarde a eviter : {sorted(deprecated_used)}"}

    known_columns = {
        c["name"] for t in referenced_tables for c in CATALOG[_SQLITE_TO_FQN[t]]["columns"]
    }
    referenced_columns = {c.name.lower() for c in tree.find_all(exp.Column)}
    unknown_columns = referenced_columns - known_columns
    if unknown_columns:
        return {"error": f"colonne(s) inconnue(s) parmi les tables utilisees : {sorted(unknown_columns)}"}

    outcome = run_sql(sql)
    if "error" in outcome:
        return {"error": outcome["error"]}
    if state["hardening"] and not outcome["rows"] and not state.get("zero_row_retried"):
        return {
            "error": "0 ligne retournee : verifie les jointures et les valeurs de filtre "
                     "(les codes sont parfois textuels, voir les valeurs observees du contexte).",
            "zero_row_retried": True,
        }
    return {"rows": outcome["rows"], "error": ""}


def route_after_validation(state: PipelineState) -> str:
    if not state.get("error"):
        return "finalize"
    if state["attempts"] >= MAX_SQL_ATTEMPTS:
        return "abstain"
    return "generate_sql"

### `finalize` / `abstain` — traçabilité de la table utilisée

`finalize` demande explicitement au modèle de citer la ou les tables utilisées dans
sa réponse — pas pour faire joli, mais parce que les questions "pièges" du
benchmark (section 7) vérifient précisément *quelle* table a servi, pas seulement
la valeur numérique.

In [ ]:
def finalize(state: PipelineState) -> dict:
    used_tables = sorted({
        _SQLITE_TO_FQN[t] for t in {tt.name.lower() for tt in sqlglot.parse_one(state["sql"], dialect="sqlite").find_all(exp.Table)}
        if t in _SQLITE_TO_FQN
    })
    response = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": (
            "Resume ces resultats SQL en une reponse courte et factuelle, en francais. "
            "Termine par la ou les tables utilisees entre parentheses."
        )},
        {"role": "user", "content": (
            f"Question : {state['question']}\nTable(s) utilisee(s) : {used_tables}\n"
            f"Resultats : {json.dumps(state['rows'], default=str)}"
        )},
    ])
    return {"answer": response.choices[0].message.content or ""}


def abstain(state: PipelineState) -> dict:
    return {
        "answer": f"abstention apres {state['attempts']} tentative(s) : {state['error']}. "
                  f"Tables envisagees : {state['shortlist']}"
    }


guided_builder = StateGraph(PipelineState)
guided_builder.add_node("retrieve_candidates", retrieve_candidates)
guided_builder.add_node("gather_context", gather_context)
guided_builder.add_node("generate_sql", generate_sql)
guided_builder.add_node("validate_and_execute", validate_and_execute)
guided_builder.add_node("finalize", finalize)
guided_builder.add_node("abstain", abstain)
guided_builder.add_edge(START, "retrieve_candidates")
guided_builder.add_edge("retrieve_candidates", "gather_context")
guided_builder.add_edge("gather_context", "generate_sql")
guided_builder.add_edge("generate_sql", "validate_and_execute")
guided_builder.add_conditional_edges(
    "validate_and_execute", route_after_validation,
    {"finalize": "finalize", "generate_sql": "generate_sql", "abstain": "abstain"},
)
guided_builder.add_edge("finalize", END)
guided_builder.add_edge("abstain", END)
guided_pipeline = guided_builder.compile()

print(guided_pipeline.get_graph().draw_mermaid())

In [ ]:
def run_guided(question: str, hardening: bool = True, verbose: bool = False) -> str:
    state: PipelineState = {
        "question": question, "hardening": hardening, "attempts": 0, "shortlist": [],
        "context": "", "sql": "", "error": "", "zero_row_retried": False, "rows": [], "answer": "",
    }
    answer = ""
    for event in guided_pipeline.stream(state, config={"recursion_limit": 25}):
        for node, update in event.items():
            if verbose:
                print(f"[{node}] {update}")
            if "answer" in update:
                answer = update["answer"]
    return answer


print("-- naif --")
print(run_guided("Quel est le solde actuel du compte 101 ?", hardening=False, verbose=True))
print("\n-- hardened --")
print(run_guided("Quel est le solde actuel du compte 101 ?", hardening=True, verbose=True))

## 6. Baseline : agent libre (sans framework)

Repris de v1 (approche A), avec les nouveaux outils (`profile_column` en plus).
Sert de point de comparaison "aucun garde-fou de conception, tout est délégué au
modèle" dans le benchmark de la section 7.

In [ ]:
SYSTEM_PROMPT = (
    "Tu es un assistant analytique sur un datamart bancaire. Le catalogue contient des "
    "tables de sauvegarde/test perimees (suffixes _BCK, _OLD, _TMP, _TEST) : ne les utilise "
    "jamais pour repondre. Utilise les outils a disposition pour explorer le catalogue, "
    "comprendre le lineage et interroger les donnees. Ne reponds jamais une valeur chiffree "
    "sans l'avoir verifiee par `run_sql`. Si les outils ne permettent pas de repondre, "
    "dis-le explicitement."
)


def run_agent_manual(question: str, verbose: bool = True) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]

    for round_ in range(MAX_TOOL_ROUNDS):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))

        if not message.tool_calls:
            return message.content or ""

        for call in message.tool_calls:
            arguments = json.loads(call.function.arguments or "{}")
            if verbose:
                print(f"[round {round_}] {call.function.name}({arguments})")
            result = dispatch_tool_call(call.function.name, arguments)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    return "abstention : nombre maximal d'allers-retours outils atteint"

## 7. Benchmark : 12 questions, 3 configurations, un LLM juge

Chaque question porte une **grille d'éléments attendus** (pas une réponse mot pour
mot) et un flag `expect_abstain` pour les deux questions dont la bonne réponse est
"cette donnée n'existe pas" — halluciner un chiffre y est un échec, pas juste une
imprécision. Les questions 1, 2 et 9 sont les pièges directs (tables `_OLD`/`_BCK`/
`_TEST` à côté de la table légitime) ; 3 et 12 exigent une jointure ; 4 et 8 exigent
le profilage d'une colonne non documentée ; 10 exige le lineage ; 6 et 11 sont hors
périmètre.

In [ ]:
BENCHMARK_QUESTIONS = [
    {"question": "Quel est le solde actuel du compte 101 ?",
     "rubric": "Doit indiquer environ 2340 CHF (dmt_cpt_sld_j), pas 999.99 (dmt_cpt_sld_j_old, table perimee)."},
    {"question": "Combien de mouvements le compte 102 a-t-il enregistres ?",
     "rubric": "3 mouvements (table dmt_cpt_mvt_j), pas les tables de sauvegarde/test."},
    {"question": "Quel est le nom et prenom du client habitant 'Rue du Lac 4' ?",
     "rubric": "Doit joindre ods_d_cli_adr et ods_d_cli pour trouver id_client=201 -> Marie Dupont."},
    {"question": "Que signifie le code d'operation 'debit' utilise dans les mouvements ?",
     "rubric": "Doit relier cd_typ_ope='debit' au libelle 'Debit' via r_typ_ope (ou un profilage equivalent)."},
    {"question": "Quel est l'encours de credit total, tous dossiers confondus ?",
     "rubric": "45000 + 128900 + 0 = 173900."},
    {"question": "Quel est le solde du compte 999 ?",
     "rubric": "Le compte 999 n'existe pas dans dmt_cpt_sld_j : doit le dire, pas inventer un montant.",
     "expect_abstain": True},
    {"question": "Quel est le taux de change de l'euro vers le franc suisse ?",
     "rubric": "0.96."},
    {"question": "Quels types d'operations existent sur les comptes, et que signifient leurs codes ?",
     "rubric": "debit/credit/virement, avec leurs libelles (Debit/Credit/Virement) via r_typ_ope."},
    {"question": "Quelle table contient l'historique reel des mouvements comptables (pas une sauvegarde ni un test) ?",
     "rubric": "Doit repondre DMT.DMT_CPT_MVT_J (ou dmt_cpt_mvt_j), pas ods_f_cpt_mvt_bck_2019 ni tec_cpt_mvt_test."},
    {"question": "D'ou vient la colonne mt_sld_chf de la table des soldes, et quelle table l'alimente ?",
     "rubric": "Doit mentionner qu'elle est calculee par cumul (SUM) des mouvements de dmt_cpt_mvt_j."},
    {"question": "Quel est le chiffre d'affaires total de l'entreprise en 2025 ?",
     "rubric": "Aucune table du catalogue ne couvre le chiffre d'affaires : doit refuser d'inventer un chiffre.",
     "expect_abstain": True},
    {"question": "Combien de clients ont un encours de credit strictement superieur a 0 ?",
     "rubric": "2 clients (201 et 202 ; 203 a un encours de 0)."},
]

### Le juge

Un modèle (potentiellement différent de l'agent — `JUDGE_MODEL_OVERRIDE`) note
chaque réponse 0/1/2 contre la grille de la question, en tenant compte du flag
`expect_abstain`. Même tolérance JSON que `generate_sql` (`_extract_json_object`).

In [ ]:
def judge(question: str, rubric: str, answer: str, expect_abstain: bool = False) -> dict:
    instructions = (
        "Tu es un evaluateur strict. On te donne une question, une grille de correction "
        "(des elements attendus, pas une reponse mot pour mot) et la reponse d'un systeme. "
        "Note 2 si la reponse satisfait la grille (ou s'abstient a bon escient si demande), "
        "1 si partiellement correcte ou incomplete, 0 si incorrecte ou si elle invente un "
        'resultat alors qu\'elle aurait du s\'abstenir. Reponds en JSON strict : '
        '{"score": 0, "reasoning": "..."}'
    )
    prompt = (
        f"Question : {question}\nGrille : {rubric}\n"
        f"Doit s'abstenir / dire qu'il n'y a pas de donnee : {expect_abstain}\n"
        f"Reponse a evaluer : {answer}"
    )
    response = judge_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "system", "content": instructions}, {"role": "user", "content": prompt}],
    )
    raw = response.choices[0].message.content or ""
    try:
        result = _extract_json_object(raw)
        return {"score": int(result["score"]), "reasoning": str(result.get("reasoning", ""))}
    except Exception as e:  # noqa: BLE001
        return {"score": 0, "reasoning": f"juge : reponse non exploitable ({e})"}

### Le harnais

`run_benchmark` exécute chaque question sur chaque approche, chronomètre, fait
juger la réponse, et n'interrompt jamais la mesure sur une panne isolée (une
approche qui plante sur une question est un résultat — un score de 0 — pas un
crash du run entier). `MAX_QUESTIONS` limite le run par défaut à un test rapide ;
passer à `len(BENCHMARK_QUESTIONS)` pour la mesure complète (attendu : plusieurs
minutes, 3 approches × 12 questions × plusieurs appels LLM chacune).

In [ ]:
def run_benchmark(approaches: dict, questions: list[dict], max_questions: int | None = None) -> list[dict]:
    results = []
    for q in questions[:max_questions]:
        for name, fn in approaches.items():
            start = time.perf_counter()
            try:
                answer = fn(q["question"])
            except Exception as e:  # noqa: BLE001 -- une panne d'approche est un resultat, pas un crash du benchmark
                answer = f"[ERREUR] {e}"
            latency = time.perf_counter() - start
            verdict = judge(q["question"], q["rubric"], answer, q.get("expect_abstain", False))
            results.append({
                "question": q["question"], "approach": name, "answer": answer,
                "latency_s": round(latency, 1), "score": verdict["score"], "reasoning": verdict["reasoning"],
            })
            print(f"[{name:15}] {q['question'][:45]:45} score={verdict['score']}  ({latency:.1f}s)")
    return results


def print_leaderboard(results: list[dict]) -> None:
    by_approach: dict[str, list[dict]] = {}
    for r in results:
        by_approach.setdefault(r["approach"], []).append(r)
    print(f"\n{'approche':16} {'score moyen /2':16} {'latence moy (s)':17} {'echecs (score=0)':16}")
    for name, rows in by_approach.items():
        mean_score = sum(r["score"] for r in rows) / len(rows)
        mean_latency = sum(r["latency_s"] for r in rows) / len(rows)
        failures = sum(1 for r in rows if r["score"] == 0)
        print(f"{name:16} {mean_score:<16.2f} {mean_latency:<17.1f} {failures}/{len(rows)}")

In [ ]:
BENCHMARK_APPROACHES = {
    "agent_libre": lambda q: run_agent_manual(q, verbose=False),
    "guide_naif": lambda q: run_guided(q, hardening=False, verbose=False),
    "guide_hardened": lambda q: run_guided(q, hardening=True, verbose=False),
}

MAX_QUESTIONS = 4  # test rapide ; passer a len(BENCHMARK_QUESTIONS) pour la mesure complete

results = run_benchmark(BENCHMARK_APPROACHES, BENCHMARK_QUESTIONS, max_questions=MAX_QUESTIONS)
print_leaderboard(results)

## 8. Lire les résultats

- **`agent_libre`** échoue typiquement sur les questions 1, 2 et 9 (pièges de
  nommage) : rien dans son prompt ni ses outils ne l'empêche de choisir la table
  au nom le plus proche, y compris une sauvegarde.
- **`guide_naif`** évite parfois ces pièges par chance (le classement sémantique
  favorise souvent la table légitime, mieux documentée), mais pas de façon fiable —
  c'est exactement ce que mesure l'écart avec `guide_hardened`.
- **`guide_hardened`** doit dominer sur les questions à piège (1, 2, 9) et sur
  celles qui nécessitent un profilage (4, 8) — c'est le test de valeur des trois
  défenses de la section 5, pas juste "le pipeline le plus élaboré gagne".

**Limites de la mesure, à ne pas ignorer** :
- 12 questions et un LLM juge donnent un signal **directionnel**, pas une preuve —
  le juge est lui-même un LLM, bruyant sur un si petit échantillon.
- Juger avec le même modèle que l'agent (`JUDGE_MODEL_OVERRIDE = None`, le défaut)
  introduit un biais d'auto-évaluation documenté dans la littérature ; pour une
  mesure sérieuse, pointer `JUDGE_MODEL_OVERRIDE` vers un modèle différent (idéalement
  plus fort) que celui évalué.
- Le catalogue de démonstration est calibré pour que les défenses de `hardening`
  gagnent — c'est un test de non-régression du *design*, pas une preuve qu'il
  gagnera dans les mêmes proportions sur votre vrai catalogue. Remplacer `CATALOG`/
  `LINEAGE_COLUMNS`/`BENCHMARK_QUESTIONS` par vos propres données pour une mesure
  qui compte vraiment.

## 9. Quoi tweaker

| Variable/fonction                | Effet                                                                 |
| ---------------------------------- | ---------------------------------------------------------------------- |
| `DEPRECATED_NAME_PATTERN`         | ajuster aux conventions réelles de votre datamart                     |
| `trust_score`                     | ajouter des signaux réels (LAST_ANALYZED, tags de classification OMD) |
| `CANDIDATE_SHORTLIST_SIZE`        | plus large = plus robuste au bruit, plus cher en tokens de contexte    |
| `MAX_SQL_ATTEMPTS`                | tolérance aux erreurs de génération avant abstention                  |
| `hardening`                       | passer `False` pour reproduire le comportement naïf de v1              |
| `CATALOG` / `LINEAGE_COLUMNS`     | remplacer par `build/catalog.json` / `build/lineage.json` du pipeline  |
| `BENCHMARK_QUESTIONS`             | remplacer par vos vraies questions métier + grilles                    |
| `JUDGE_MODEL_OVERRIDE`            | pointer vers un modèle différent de l'agent pour limiter le biais      |
| `MAX_QUESTIONS`                   | `None` ou `len(BENCHMARK_QUESTIONS)` pour la mesure complète           |